# EDA on silver layer - validate the cleaning

Bronze EDA explored raw data to design the cleaning. This notebook *validates* that `transformations/silver/marathon_obt.py`.

In [0]:
silver = spark.read.table("marathos.silver.marathon_obt")

display(silver)

In [0]:
print(f"Number of rows {silver.count()}")
print(f"Number of columns {len(silver.columns)}")

In [0]:
silver.printSchema()

## Bronze -> Silver row loss

Total rows dropped across all cleaning filters.

In [0]:
bronze_count = spark.read.table("marathos.bronze.raw_marathos").count()
silver_count = silver.count()
dropped = bronze_count - silver_count

print(f"Bronze:  {bronze_count:,}")
print(f"Silver:  {silver_count:,}")
print(f"Dropped: {dropped:,} ({dropped / bronze_count * 100:.2f}%)")

## Did invalid units get dropped?

Only `km`, `mi`, `h` should remain. No `d` (days), no empty/unknown units.

In [0]:
from pyspark.sql.functions import col, count, desc

silver.groupBy("event_unit").count().orderBy(desc("count")).display()

## Year scope: only 2000+ should remain

In [0]:
from pyspark.sql.functions import min as spark_min, max as spark_max

silver.select(
    spark_min("year_of_event").alias("min_year"),
    spark_max("year_of_event").alias("max_year")
).display()

## Performance parsing check

Distance events (km/mi) should have `performance_seconds` populated and `performance_km` null.
Duration events (h) should be the reverse. Each row should have exactly one populated.

In [0]:
silver.groupBy("event_type").agg(
    count("*").alias("rows"),
    count("performance_seconds").alias("has_seconds"),
    count("performance_km").alias("has_km"),
).display()

In [0]:
silver.filter(col("event_type") == "distance") \
      .select("event_name", "event_unit", "event_value", "performance_seconds", "performance_km") \
      .show(5, truncate=False)

In [0]:
silver.filter(col("event_type") == "duration") \
      .select("event_name", "event_unit", "event_value", "performance_seconds", "performance_km") \
      .show(5, truncate=False)

## Stage races dropped?

Should be 0 rows containing `etappe`, `etape`, `étape`, or `stage N` in `event_name`.

In [0]:
from pyspark.sql.functions import lower

stage_rows = silver.filter(
    lower(col("event_name")).rlike(r"etappe|etape|étape|\bstage\s+[0-9]")
).count()

print(f"Stage race rows remaining: {stage_rows}")
# Should be 0

## Unidentified athletes dropped?

Should be 0 rows with `athlete_country = 'XXX'` or null `athlete_year_of_birth`.

In [0]:
xxx_rows = silver.filter(col("athlete_country") == "XXX").count()
null_byear = silver.filter(col("athlete_year_of_birth").isNull()).count()

print(f"XXX country rows:        {xxx_rows}")
print(f"Null birth year rows:    {null_byear}")
# Both should be 0

## Avg speed check

Source `athlete_average_speed` had garbage (10000+ km/h). We recomputed and dropped speeds >30 km/h and = 0. Ranges should now be physical:
- min > 0 (no DNS/DNF zeros)
- avg around 7-11 km/h (ultra pace, slower than marathon)
- max <= 30 km/h

In [0]:
silver.select("athlete_avg_speed_kmh").summary().display()

In [0]:
zero_speeds = silver.filter(col("athlete_avg_speed_kmh") == 0).count()
print(f"Rows with zero speed: {zero_speeds}")
# Should be 0

## Surrogate IDs look reasonable?

- `event_id` should be stable - one id per unique `event_name`.
- `athlete_id` count gives a realistic number of unique athletes.
- `result_id` should be unique (or near-unique) per row.

In [0]:
print(f"Unique event_names:   {silver.select('event_name').distinct().count():,}")
print(f"Unique event_ids:     {silver.select('event_id').distinct().count():,}")
print(f"Unique athlete_ids:   {silver.select('athlete_id').distinct().count():,}")
print(f"Unique result_ids:    {silver.select('result_id').distinct().count():,}")
print(f"Total rows:           {silver.count():,}")

## Final Check

Check how many (athlete_id, event_name, event_dates) groups have more than one row. Some duplicates are expected since different athletes can share the same country, gender, birth year, and source_id.

In [0]:
duplicate_groups = (silver.groupBy("athlete_id", "event_name", "event_dates")
                    .count()
                    .filter(col("count") > 1)
                    .count())

total = silver.count()
print(f"Duplicate groups: {duplicate_groups:,}")
print(f"Total rows:       {total:,}")
print(f"Pct duplicates:   {duplicate_groups / total * 100:.3f}%")

## Null check on the cleaned table

Expected nulls: `performance_seconds` is null for duration rows and `performance_km` is null for distance rows (by design).

In [0]:
from pyspark.sql.functions import sum as spark_sum

null_counts = silver.select(
    [spark_sum(col(c).isNull().cast("int")).alias(c) for c in silver.columns]
)
null_counts = null_counts.collect()[0].asDict()
[(c, n) for c, n in null_counts.items() if n > 0]

## Performance distribution - 50km finish times (Plotly)

Check finish times for 50km races. Should peak around 5-7 hours. A wrong peak means the seconds parsing is broken.

In [0]:
import matplotlib.pyplot as plt

times["hours"].plot.hist(bins=40, title="50km finish times (hours)")
plt.xlabel("Hours")
plt.show()

## Avg speed distribution (Plotly)

Visual check on the recomputed speed - should be a smooth distribution capped at 30 km/h.

In [0]:
speeds = (
    silver.select("athlete_avg_speed_kmh")
    .filter(col("athlete_avg_speed_kmh").isNotNull())
    .sample(0.05, seed=42)
    .toPandas()
)

fig = px.histogram(speeds, x="athlete_avg_speed_kmh", nbins=60,
                   title="Athlete average speed distribution (5% sample)")
fig.show()

## Conclusion

The silver table looks clean and ready to use.

**Structure**
- Columns renamed to snake_case, event distance split into value and unit, and performance parsed into typed columns.

**Filters**
- Only km, mi, and h events remain.
- Data is scoped to 2000-2022.
- No stage races, unidentified athletes, or implausible speeds remaining.
- A small number of duplicate residuals (~0.01%) are kept and documented as a known limitation.

**Checks**
- Every row has either `performance_seconds` or `performance_km` filled, never both.
- 50km finish times peak around 6 hours which looks realistic.
- Average speed around 7-8 km/h which makes sense for ultra running.

Ready for gold.
